# Hinge loss SVM

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATH = r"D:\墨大sml作业\FeatureA_Repeated"
OUTPUT_PATH = r"D:\墨大sml作业\SVM_FeatureA_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

# SVM hyperparameter candidates
# alpha is the regularization strength in SGDClassifier
ALPHA_VALUES = [1e-5, 1e-4, 1e-3]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2
BASE_SEED = 42

In [3]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [4]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)
    train_indices = []
    test_indices = []
    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)
        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [5]:
def compute_basic_metrics_from_scratch(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [6]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    sorted_indices = np.argsort(-y_score)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            *
            (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc

In [7]:
def train_and_evaluate_hinge_svm(train_df, test_df, alpha_value):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols]
    y_train = train_df["label"]

    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    svm_model = SGDClassifier(
        loss="hinge",
        penalty="l2",
        alpha=alpha_value,
        max_iter=1000,
        tol=1e-3,
        random_state=42
    )

    svm_model.fit(X_train_scaled, y_train)

    y_pred = svm_model.predict(X_test_scaled)

    # For hinge-loss SVM, decision_function gives the margin score
    y_score = svm_model.decision_function(X_test_scaled)

    metrics = compute_basic_metrics_from_scratch(y_test, y_pred)
    metrics["auc"] = compute_auc_from_scratch(y_test, y_score)

    return metrics

In [8]:
def tune_svm_alpha_from_scratch(
    train_df,
    alpha_values,
    n_inner_repeats=3,
    valid_ratio=0.2,
    base_seed=100
):
    tuning_records = []

    for alpha_value in alpha_values:
        inner_f1_scores = []

        for inner_id in range(n_inner_repeats):
            inner_train_df, valid_df = stratified_split_from_scratch(
                train_df,
                label_col="label",
                test_ratio=valid_ratio,
                random_seed=base_seed + inner_id
            )

            metrics = train_and_evaluate_hinge_svm(
                train_df=inner_train_df,
                test_df=valid_df,
                alpha_value=alpha_value
            )

            inner_f1_scores.append(metrics["f1"])

        tuning_records.append({
            "alpha": alpha_value,
            "mean_validation_f1": np.mean(inner_f1_scores),
            "std_validation_f1": np.std(inner_f1_scores, ddof=1)
        })

    tuning_df = pd.DataFrame(tuning_records)

    best_alpha = tuning_df.sort_values(
        by="mean_validation_f1",
        ascending=False
    ).iloc[0]["alpha"]

    return best_alpha, tuning_df

In [9]:
all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):

    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_A_train.csv")
    test_path = os.path.join(repeat_folder, "feature_A_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)

    best_alpha, tuning_df = tune_svm_alpha_from_scratch(
        train_df=train_df,
        alpha_values=ALPHA_VALUES,
        n_inner_repeats=N_INNER_REPEATS,
        valid_ratio=VALID_RATIO,
        base_seed=2000 + repeat_id * 10
    )

    print("Best alpha:", best_alpha)

    tuning_df["outer_repeat"] = repeat_id
    all_tuning_results.append(tuning_df)

    final_metrics = train_and_evaluate_hinge_svm(
        train_df=train_df,
        test_df=test_df,
        alpha_value=best_alpha
    )

    result_row = {
        "outer_repeat": repeat_id,
        "best_alpha": best_alpha,
        **final_metrics
    }

    all_results.append(result_row)

    print("Accuracy :", round(final_metrics["accuracy"], 4))
    print("Precision:", round(final_metrics["precision"], 4))
    print("Recall   :", round(final_metrics["recall"], 4))
    print("F1       :", round(final_metrics["f1"], 4))
    print("AUC      :", round(final_metrics["auc"], 4))

Outer Repeat 01
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best alpha: 1e-05
Accuracy : 0.7136
Precision: 0.7072
Recall   : 0.7285
F1       : 0.7177
AUC      : 0.7871
Outer Repeat 02
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best alpha: 0.001
Accuracy : 0.714
Precision: 0.708
Recall   : 0.7279
F1       : 0.7178
AUC      : 0.7877
Outer Repeat 03
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best alpha: 0.001
Accuracy : 0.7144
Precision: 0.7083
Recall   : 0.7286
F1       : 0.7183
AUC      : 0.7879
Outer Repeat 04
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best alpha: 0.0001
Accuracy : 0.7139
Precision: 0.7059
Recall   : 0.7326
F1       : 0.719
AUC      : 0.7874
Outer Repeat 05
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best alpha: 0.001
Accuracy : 0.7137
Precision: 0.7081
Recall   : 0.7269
F1       : 0.7174
AUC      : 0.7874
Outer Repeat 06
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best alpha: 0.001
Accuracy : 0.714
Precisi

C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
C:\Users\MJ\AppData\Roaming\Python\Python39\site-packages\sklearn\preprocessing\_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=

Best alpha: 1e-05
Accuracy : 0.7123
Precision: 0.7094
Recall   : 0.7186
F1       : 0.714
AUC      : 0.7861
Outer Repeat 10
Train shape: (16000211, 17)
Test shape: (4000052, 17)
Best alpha: 0.001
Accuracy : 0.7141
Precision: 0.7086
Recall   : 0.7269
F1       : 0.7177
AUC      : 0.7875


In [10]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.concat(all_tuning_results, ignore_index=True)

results_path = os.path.join(OUTPUT_PATH, "SVM_FeatureA_repeated_results.csv")
tuning_path = os.path.join(OUTPUT_PATH, "SVM_FeatureA_tuning_results.csv")

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
D:\墨大sml作业\SVM_FeatureA_Results\SVM_FeatureA_repeated_results.csv
Saved tuning results to:
D:\墨大sml作业\SVM_FeatureA_Results\SVM_FeatureA_tuning_results.csv


,outer_repeat,best_alpha,accuracy,precision,recall,f1,tp,tn,fp,fn,auc
0,1,0.00001,0.713618,0.707248,0.728524,0.717728,1456379,1398129,602841,542703,0.787125
1,2,0.00100,0.714003,0.708036,0.727885,0.717823,1455102,1400948,600022,543980,0.787660
2,3,0.00100,0.714395,0.708298,0.728572,0.718292,1456475,1401141,599829,542607,0.787944
3,4,0.00010,0.713851,0.705940,0.732595,0.719021,1464518,1390925,610045,534564,0.787441
4,5,0.00100,0.713746,0.708094,0.726868,0.717358,1453069,1401953,599017,546013,0.787394
5,6,0.00100,0.714033,0.708336,0.727245,0.717666,1453823,1402345,598625,545259,0.787676
6,7,0.00001,0.713334,0.709702,0.721536,0.715570,1442410,1410964,590006,556672,0.787014
7,8,0.00100,0.714021,0.707330,0.729698,0.718340,1458726,1397397,603573,540356,0.787757
8,9,0.00001,0.712259,0.709420,0.718576,0.713969,1436492,1412581,588389,562590,0.786107
9,10,0.00100,0.714140,0.708605,0.726948,0.717659,1453228,1403370,597600,545854,0.787526


In [11]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(OUTPUT_PATH, "SVM_FeatureA_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

summary_df

,metric,mean,std,standard_error
0,accuracy,0.713740,0.000597,0.000189
1,precision,0.708101,0.001088,0.000344
2,recall,0.726845,0.004011,0.001269
3,f1,0.717343,0.001484,0.000469
4,auc,0.787364,0.000524,0.000166


In [12]:
best_alpha_frequency = results_df["best_alpha"].value_counts().reset_index()
best_alpha_frequency.columns = ["alpha", "frequency"]

best_alpha_frequency_path = os.path.join(
    OUTPUT_PATH,
    "SVM_FeatureA_best_alpha_frequency.csv"
)
best_alpha_frequency.to_csv(best_alpha_frequency_path, index=False, encoding="utf-8-sig")

best_alpha_frequency

,alpha,frequency
0,0.00100,6
1,0.00001,3
2,0.00010,1
